# HR Employee Management System Using RDBMS Concepts

## 1. Business understanding and schema design

### Main entities
- `departments`
- `job_roles`
- `employees`
- `attendance`
- `payroll`

### Relationships
- One department can have many employees.
- One job role can be assigned to many employees.
- One employee can have many attendance records.
- One employee can have many payroll records across different months.

### Constraints applied
- Primary keys on all tables.
- Foreign keys from employees to departments and job_roles.
- Foreign keys from attendance and payroll to employees.
- Unique keys on department name, role title, and employee email.
- Composite unique constraints on `(employee_id, attendance_date)` and `(employee_id, salary_month)`.
- `NOT NULL`, `CHECK`, and `DEFAULT` constraints as required.


In [1]:
import sqlite3
from pprint import pprint

conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys = ON;')
cur = conn.cursor()

def run(query, params=None, fetch=True, many=False):
    if params is None:
        params = []
    if many:
        cur.executemany(query, params)
    else:
        cur.execute(query, params)
    conn.commit()
    if fetch:
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description] if cur.description else []
        print(cols)
        for r in rows:
            print(r)

def try_run(label, query, params=None):
    try:
        cur.execute(query, params or [])
        conn.commit()
        print(f'{label}: SUCCESS')
    except Exception as e:
        conn.rollback()
        print(f'{label}: FAILED -> {e}')


## 2. Create the tables


In [2]:
create_departments = '''
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL UNIQUE,
    CHECK (trim(department_name) <> '')
);
'''

create_job_roles = '''
CREATE TABLE job_roles (
    role_id INTEGER PRIMARY KEY,
    role_title TEXT NOT NULL UNIQUE,
    min_salary REAL NOT NULL CHECK (min_salary >= 0),
    max_salary REAL NOT NULL CHECK (max_salary >= min_salary),
    CHECK (trim(role_title) <> '')
);
'''

create_employees = '''
CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    city TEXT NOT NULL,
    joining_date DATE NOT NULL,
    department_id INTEGER NOT NULL,
    role_id INTEGER NOT NULL,
    status TEXT NOT NULL DEFAULT 'Active',
    CHECK (trim(employee_name) <> ''),
    CHECK (trim(email) <> ''),
    CHECK (status IN ('Active', 'Probation', 'Resigned')),
    FOREIGN KEY (department_id) REFERENCES departments(department_id),
    FOREIGN KEY (role_id) REFERENCES job_roles(role_id)
);
'''

create_attendance = '''
CREATE TABLE attendance (
    attendance_id INTEGER PRIMARY KEY,
    employee_id INTEGER NOT NULL,
    attendance_date DATE NOT NULL,
    attendance_status TEXT NOT NULL,
    CHECK (attendance_status IN ('Present', 'Absent', 'Leave', 'Half Day')),
    UNIQUE (employee_id, attendance_date),
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
);
'''

create_payroll = '''
CREATE TABLE payroll (
    payroll_id INTEGER PRIMARY KEY,
    employee_id INTEGER NOT NULL,
    salary_month TEXT NOT NULL,
    basic_salary REAL NOT NULL CHECK (basic_salary >= 0),
    bonus REAL NOT NULL DEFAULT 0 CHECK (bonus >= 0),
    deductions REAL NOT NULL DEFAULT 0 CHECK (deductions >= 0),
    payment_status TEXT NOT NULL,
    CHECK (payment_status IN ('Paid', 'Pending', 'Hold')),
    UNIQUE (employee_id, salary_month),
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
);
'''

for ddl in [create_departments, create_job_roles, create_employees, create_attendance, create_payroll]:
    cur.execute(ddl)
conn.commit()
print('All five tables created successfully.')


All five tables created successfully.


## 3. Insert sample data

Data is inserted in the correct parent-to-child order: departments, job_roles, employees, attendance, payroll.


In [3]:
departments_data = [
    (1, 'Human Resources'),
    (2, 'Information Technology'),
    (3, 'Finance'),
    (4, 'Sales'),
    (5, 'Operations')
]

job_roles_data = [
    (1, 'HR Executive', 25000, 50000),
    (2, 'Software Developer', 40000, 120000),
    (3, 'Accountant', 30000, 70000),
    (4, 'Sales Executive', 25000, 60000),
    (5, 'Operations Associate', 22000, 50000)
]

employees_data = [
    (1, 'Rahul Kumar', 'rahul.kumar@example.com', 'Patna', '2024-06-10', 1, 1, 'Active'),
    (2, 'Priya Singh', 'priya.singh@example.com', 'Kolkata', '2023-11-01', 2, 2, 'Active'),
    (3, 'Amit Raj', 'amit.raj@example.com', 'Delhi', '2026-01-20', 3, 3, 'Probation'),
    (4, 'Sneha Verma', 'sneha.verma@example.com', 'Patna', '2024-09-15', 4, 4, 'Active'),
    (5, 'Aditya Sharma', 'aditya.sharma@example.com', 'Mumbai', '2025-03-12', 5, 5, 'Active')
]

attendance_data = [
    (1, 1, '2026-02-01', 'Present'),
    (2, 2, '2026-02-01', 'Present'),
    (3, 3, '2026-02-01', 'Leave'),
    (4, 4, '2026-02-01', 'Half Day'),
    (5, 5, '2026-02-01', 'Absent')
]

payroll_data = [
    (1, 1, '2026-02', 35000, 2000, 1000, 'Paid'),
    (2, 2, '2026-02', 80000, 5000, 2000, 'Paid'),
    (3, 3, '2026-02', 45000, 0, 1000, 'Pending'),
    (4, 4, '2026-02', 30000, 3000, 500, 'Paid'),
    (5, 5, '2026-02', 28000, 0, 0, 'Hold')
]

run('INSERT INTO departments VALUES (?, ?)', departments_data, fetch=False, many=True)
run('INSERT INTO job_roles VALUES (?, ?, ?, ?)', job_roles_data, fetch=False, many=True)
run('INSERT INTO employees VALUES (?, ?, ?, ?, ?, ?, ?, ?)', employees_data, fetch=False, many=True)
run('INSERT INTO attendance VALUES (?, ?, ?, ?)', attendance_data, fetch=False, many=True)
run('INSERT INTO payroll VALUES (?, ?, ?, ?, ?, ?, ?)', payroll_data, fetch=False, many=True)
print('Sample data inserted successfully.')


Sample data inserted successfully.


## 4. Display all table data


In [4]:
print('DEPARTMENTS')
run('SELECT * FROM departments')
print('\nJOB ROLES')
run('SELECT * FROM job_roles')
print('\nEMPLOYEES')
run('SELECT * FROM employees')
print('\nATTENDANCE')
run('SELECT * FROM attendance')
print('\nPAYROLL')
run('SELECT * FROM payroll')


DEPARTMENTS
['department_id', 'department_name']
(1, 'Human Resources')
(2, 'Information Technology')
(3, 'Finance')
(4, 'Sales')
(5, 'Operations')

JOB ROLES
['role_id', 'role_title', 'min_salary', 'max_salary']
(1, 'HR Executive', 25000.0, 50000.0)
(2, 'Software Developer', 40000.0, 120000.0)
(3, 'Accountant', 30000.0, 70000.0)
(4, 'Sales Executive', 25000.0, 60000.0)
(5, 'Operations Associate', 22000.0, 50000.0)

EMPLOYEES
['employee_id', 'employee_name', 'email', 'city', 'joining_date', 'department_id', 'role_id', 'status']
(1, 'Rahul Kumar', 'rahul.kumar@example.com', 'Patna', '2024-06-10', 1, 1, 'Active')
(2, 'Priya Singh', 'priya.singh@example.com', 'Kolkata', '2023-11-01', 2, 2, 'Active')
(3, 'Amit Raj', 'amit.raj@example.com', 'Delhi', '2026-01-20', 3, 3, 'Probation')
(4, 'Sneha Verma', 'sneha.verma@example.com', 'Patna', '2024-09-15', 4, 4, 'Active')
(5, 'Aditya Sharma', 'aditya.sharma@example.com', 'Mumbai', '2025-03-12', 5, 5, 'Active')

ATTENDANCE
['attendance_id', 'employ

## 5. Basic SQL queries


In [5]:
print('1) Show all employees')
run('SELECT * FROM employees')

print('\n2) Show employee name, email, and city')
run('SELECT employee_name, email, city FROM employees')

print('\n3) Show all employees from Patna')
run("SELECT * FROM employees WHERE city = 'Patna'")

print('\n4) Show all active employees')
run("SELECT * FROM employees WHERE status = 'Active'")

print('\n5) Show all employees under probation')
run("SELECT * FROM employees WHERE status = 'Probation'")

print('\n6) Show all employees sorted by joining date')
run('SELECT * FROM employees ORDER BY joining_date')

print('\n7) Show the top 3 highest-paid employees')
run('''
SELECT e.employee_name, p.salary_month, p.basic_salary, p.bonus, p.deductions,
       (p.basic_salary + p.bonus - p.deductions) AS net_salary
FROM payroll p
JOIN employees e ON e.employee_id = p.employee_id
ORDER BY net_salary DESC
LIMIT 3
''')

print('\n8) Show all distinct employee cities')
run('SELECT DISTINCT city FROM employees')

print('\n9) Show payroll records where payment status is Pending')
run("SELECT * FROM payroll WHERE payment_status = 'Pending'")

print('\n10) Show attendance records where attendance status is Absent')
run("SELECT * FROM attendance WHERE attendance_status = 'Absent'")


1) Show all employees
['employee_id', 'employee_name', 'email', 'city', 'joining_date', 'department_id', 'role_id', 'status']
(1, 'Rahul Kumar', 'rahul.kumar@example.com', 'Patna', '2024-06-10', 1, 1, 'Active')
(2, 'Priya Singh', 'priya.singh@example.com', 'Kolkata', '2023-11-01', 2, 2, 'Active')
(3, 'Amit Raj', 'amit.raj@example.com', 'Delhi', '2026-01-20', 3, 3, 'Probation')
(4, 'Sneha Verma', 'sneha.verma@example.com', 'Patna', '2024-09-15', 4, 4, 'Active')
(5, 'Aditya Sharma', 'aditya.sharma@example.com', 'Mumbai', '2025-03-12', 5, 5, 'Active')

2) Show employee name, email, and city
['employee_name', 'email', 'city']
('Rahul Kumar', 'rahul.kumar@example.com', 'Patna')
('Priya Singh', 'priya.singh@example.com', 'Kolkata')
('Amit Raj', 'amit.raj@example.com', 'Delhi')
('Sneha Verma', 'sneha.verma@example.com', 'Patna')
('Aditya Sharma', 'aditya.sharma@example.com', 'Mumbai')

3) Show all employees from Patna
['employee_id', 'employee_name', 'email', 'city', 'joining_date', 'departme

## 6. Insert, update, and delete operations


In [6]:
print('Insert Rohan Das')
run('''
INSERT INTO employees (employee_id, employee_name, email, city, joining_date, department_id, role_id, status)
VALUES (6, 'Rohan Das', 'rohan.hr@example.com', 'Pune', '2026-02-15',
        (SELECT department_id FROM departments WHERE department_name = 'Human Resources'),
        (SELECT role_id FROM job_roles WHERE role_title = 'HR Executive'),
        'Probation')
''', fetch=False)
run("SELECT * FROM employees WHERE employee_name = 'Rohan Das'")

print('\nUpdate city from Pune to Bengaluru')
run("UPDATE employees SET city = 'Bengaluru' WHERE employee_name = 'Rohan Das'", fetch=False)
run("SELECT * FROM employees WHERE employee_name = 'Rohan Das'")

print('\nUpdate status from Probation to Active')
run("UPDATE employees SET status = 'Active' WHERE employee_name = 'Rohan Das'", fetch=False)
run("SELECT * FROM employees WHERE employee_name = 'Rohan Das'")

print('\nDelete Rohan Das only if no attendance/payroll records exist')
run('''
DELETE FROM employees
WHERE employee_name = 'Rohan Das'
  AND employee_id NOT IN (SELECT employee_id FROM attendance)
  AND employee_id NOT IN (SELECT employee_id FROM payroll)
''', fetch=False)
run('SELECT * FROM employees ORDER BY employee_id')


Insert Rohan Das
['employee_id', 'employee_name', 'email', 'city', 'joining_date', 'department_id', 'role_id', 'status']
(6, 'Rohan Das', 'rohan.hr@example.com', 'Pune', '2026-02-15', 1, 1, 'Probation')

Update city from Pune to Bengaluru
['employee_id', 'employee_name', 'email', 'city', 'joining_date', 'department_id', 'role_id', 'status']
(6, 'Rohan Das', 'rohan.hr@example.com', 'Bengaluru', '2026-02-15', 1, 1, 'Probation')

Update status from Probation to Active
['employee_id', 'employee_name', 'email', 'city', 'joining_date', 'department_id', 'role_id', 'status']
(6, 'Rohan Das', 'rohan.hr@example.com', 'Bengaluru', '2026-02-15', 1, 1, 'Active')

Delete Rohan Das only if no attendance/payroll records exist
['employee_id', 'employee_name', 'email', 'city', 'joining_date', 'department_id', 'role_id', 'status']
(1, 'Rahul Kumar', 'rahul.kumar@example.com', 'Patna', '2024-06-10', 1, 1, 'Active')
(2, 'Priya Singh', 'priya.singh@example.com', 'Kolkata', '2023-11-01', 2, 2, 'Active')
(3, 

## 7. Constraint testing

Each invalid operation below should fail if the constraints are working correctly.


In [7]:
try_run('Test 1 Duplicate Email', '''
INSERT INTO employees (employee_id, employee_name, email, city, joining_date, department_id, role_id, status)
VALUES (7, 'Duplicate Email User', 'rahul.kumar@example.com', 'Noida', '2026-03-01', 1, 1, 'Active')
''')

try_run('Test 2 Invalid Department', '''
INSERT INTO employees (employee_id, employee_name, email, city, joining_date, department_id, role_id, status)
VALUES (8, 'Invalid Dept User', 'invalid.dept@example.com', 'Noida', '2026-03-01', 99, 1, 'Active')
''')

try_run('Test 3 Invalid Job Role', '''
INSERT INTO employees (employee_id, employee_name, email, city, joining_date, department_id, role_id, status)
VALUES (9, 'Invalid Role User', 'invalid.role@example.com', 'Noida', '2026-03-01', 1, 99, 'Active')
''')

try_run('Test 4 Invalid Employee Status', '''
INSERT INTO employees (employee_id, employee_name, email, city, joining_date, department_id, role_id, status)
VALUES (10, 'Joined User', 'joined.user@example.com', 'Noida', '2026-03-01', 1, 1, 'Joined')
''')

try_run('Test 5 Duplicate Attendance first insert', '''
INSERT INTO attendance (attendance_id, employee_id, attendance_date, attendance_status)
VALUES (6, 1, '2026-02-02', 'Present')
''')
try_run('Test 5 Duplicate Attendance second insert', '''
INSERT INTO attendance (attendance_id, employee_id, attendance_date, attendance_status)
VALUES (7, 1, '2026-02-02', 'Absent')
''')

try_run('Test 6 Invalid Attendance Status', '''
INSERT INTO attendance (attendance_id, employee_id, attendance_date, attendance_status)
VALUES (8, 2, '2026-02-02', 'Late')
''')

try_run('Test 7 Duplicate Payroll first insert', '''
INSERT INTO payroll (payroll_id, employee_id, salary_month, basic_salary, bonus, deductions, payment_status)
VALUES (6, 1, '2026-03', 36000, 1000, 500, 'Paid')
''')
try_run('Test 7 Duplicate Payroll second insert', '''
INSERT INTO payroll (payroll_id, employee_id, salary_month, basic_salary, bonus, deductions, payment_status)
VALUES (7, 1, '2026-03', 36000, 500, 100, 'Paid')
''')

try_run('Test 8 Negative Salary', '''
INSERT INTO payroll (payroll_id, employee_id, salary_month, basic_salary, bonus, deductions, payment_status)
VALUES (8, 2, '2026-03', -1000, 0, 0, 'Paid')
''')

try_run('Test 9 Invalid Payroll Status', '''
INSERT INTO payroll (payroll_id, employee_id, salary_month, basic_salary, bonus, deductions, payment_status)
VALUES (9, 2, '2026-03', 85000, 1000, 500, 'Processing')
''')


Test 1 Duplicate Email: FAILED -> UNIQUE constraint failed: employees.email
Test 2 Invalid Department: FAILED -> FOREIGN KEY constraint failed
Test 3 Invalid Job Role: FAILED -> FOREIGN KEY constraint failed
Test 4 Invalid Employee Status: FAILED -> CHECK constraint failed: status IN ('Active', 'Probation', 'Resigned')
Test 5 Duplicate Attendance first insert: SUCCESS
Test 5 Duplicate Attendance second insert: FAILED -> UNIQUE constraint failed: attendance.employee_id, attendance.attendance_date
Test 6 Invalid Attendance Status: FAILED -> CHECK constraint failed: attendance_status IN ('Present', 'Absent', 'Leave', 'Half Day')
Test 7 Duplicate Payroll first insert: SUCCESS
Test 7 Duplicate Payroll second insert: FAILED -> UNIQUE constraint failed: payroll.employee_id, payroll.salary_month
Test 8 Negative Salary: FAILED -> CHECK constraint failed: basic_salary >= 0
Test 9 Invalid Payroll Status: FAILED -> CHECK constraint failed: payment_status IN ('Paid', 'Pending', 'Hold')


## 8. Schema inspection

SQLite schema can be inspected using `PRAGMA table_info`, `PRAGMA foreign_key_list`, and the table creation SQL from `sqlite_master`.


In [8]:
tables = ['departments', 'job_roles', 'employees', 'attendance', 'payroll']
for table in tables:
    print(f'\n===== {table.upper()} : table_info =====')
    run(f'PRAGMA table_info({table})')
    print(f'\n===== {table.upper()} : foreign_key_list =====')
    run(f'PRAGMA foreign_key_list({table})')
    print(f'\n===== {table.upper()} : create SQL =====')
    run(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table}'")



===== DEPARTMENTS : table_info =====
['cid', 'name', 'type', 'notnull', 'dflt_value', 'pk']
(0, 'department_id', 'INTEGER', 0, None, 1)
(1, 'department_name', 'TEXT', 1, None, 0)

===== DEPARTMENTS : foreign_key_list =====
['id', 'seq', 'table', 'from', 'to', 'on_update', 'on_delete', 'match']

===== DEPARTMENTS : create SQL =====
['sql']
("CREATE TABLE departments (\n    department_id INTEGER PRIMARY KEY,\n    department_name TEXT NOT NULL UNIQUE,\n    CHECK (trim(department_name) <> '')\n)",)

===== JOB_ROLES : table_info =====
['cid', 'name', 'type', 'notnull', 'dflt_value', 'pk']
(0, 'role_id', 'INTEGER', 0, None, 1)
(1, 'role_title', 'TEXT', 1, None, 0)
(2, 'min_salary', 'REAL', 1, None, 0)
(3, 'max_salary', 'REAL', 1, None, 0)

===== JOB_ROLES : foreign_key_list =====
['id', 'seq', 'table', 'from', 'to', 'on_update', 'on_delete', 'match']

===== JOB_ROLES : create SQL =====
['sql']
("CREATE TABLE job_roles (\n    role_id INTEGER PRIMARY KEY,\n    role_title TEXT NOT NULL UNIQUE,

## 9. Short explanation of table relationships

- `departments.department_id` is referenced by `employees.department_id`, so one department can have many employees.
- `job_roles.role_id` is referenced by `employees.role_id`, so one role can be assigned to many employees.
- `employees.employee_id` is referenced by `attendance.employee_id`, so one employee can have many attendance rows.
- `employees.employee_id` is referenced by `payroll.employee_id`, so one employee can have many payroll rows across salary months.
- Duplicate attendance and payroll rows are prevented by composite unique constraints.


## 10. SQL script only version

The following cell stores the complete SQL solution as a Python multiline string so it can also be reused separately if needed.


In [9]:
sql_solution = '''
-- CREATE TABLES
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL UNIQUE,
    CHECK (trim(department_name) <> '')
);

CREATE TABLE job_roles (
    role_id INTEGER PRIMARY KEY,
    role_title TEXT NOT NULL UNIQUE,
    min_salary REAL NOT NULL CHECK (min_salary >= 0),
    max_salary REAL NOT NULL CHECK (max_salary >= min_salary),
    CHECK (trim(role_title) <> '')
);

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    city TEXT NOT NULL,
    joining_date DATE NOT NULL,
    department_id INTEGER NOT NULL,
    role_id INTEGER NOT NULL,
    status TEXT NOT NULL DEFAULT 'Active',
    CHECK (trim(employee_name) <> ''),
    CHECK (trim(email) <> ''),
    CHECK (status IN ('Active', 'Probation', 'Resigned')),
    FOREIGN KEY (department_id) REFERENCES departments(department_id),
    FOREIGN KEY (role_id) REFERENCES job_roles(role_id)
);

CREATE TABLE attendance (
    attendance_id INTEGER PRIMARY KEY,
    employee_id INTEGER NOT NULL,
    attendance_date DATE NOT NULL,
    attendance_status TEXT NOT NULL,
    CHECK (attendance_status IN ('Present', 'Absent', 'Leave', 'Half Day')),
    UNIQUE (employee_id, attendance_date),
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
);

CREATE TABLE payroll (
    payroll_id INTEGER PRIMARY KEY,
    employee_id INTEGER NOT NULL,
    salary_month TEXT NOT NULL,
    basic_salary REAL NOT NULL CHECK (basic_salary >= 0),
    bonus REAL NOT NULL DEFAULT 0 CHECK (bonus >= 0),
    deductions REAL NOT NULL DEFAULT 0 CHECK (deductions >= 0),
    payment_status TEXT NOT NULL,
    CHECK (payment_status IN ('Paid', 'Pending', 'Hold')),
    UNIQUE (employee_id, salary_month),
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
);
'''
print(sql_solution)



-- CREATE TABLES
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL UNIQUE,
    CHECK (trim(department_name) <> '')
);

CREATE TABLE job_roles (
    role_id INTEGER PRIMARY KEY,
    role_title TEXT NOT NULL UNIQUE,
    min_salary REAL NOT NULL CHECK (min_salary >= 0),
    max_salary REAL NOT NULL CHECK (max_salary >= min_salary),
    CHECK (trim(role_title) <> '')
);

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    city TEXT NOT NULL,
    joining_date DATE NOT NULL,
    department_id INTEGER NOT NULL,
    role_id INTEGER NOT NULL,
    status TEXT NOT NULL DEFAULT 'Active',
    CHECK (trim(employee_name) <> ''),
    CHECK (trim(email) <> ''),
    CHECK (status IN ('Active', 'Probation', 'Resigned')),
    FOREIGN KEY (department_id) REFERENCES departments(department_id),
    FOREIGN KEY (role_id) REFERENCES job_roles(role_id)
);

CREATE TABLE attendance (
 